# <font color="#418FDE" size="6.5" uppercase>**Optimierung verstehen**</font>

>Last update: 20260723.
    
By the end of this Lecture, you will be able to:
- Beschreiben eine Verlustfunktion für einen einzelnen Modellparameter. 
- Implementieren Raster- und Gradientenabstiegssuche für einfache Funktionen. 
- Analysieren Lernrate, Abbruchbedingungen, Skalierung und Verlustkurven. 


## **1. Parameter und Verlust**

### **1.1. Ein Parameter**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_07/Lecture_A/image_01_01.jpg?v=1784790285" width="250">



>* Ein Parameter steuert die Modellvorhersagen
>* Optimierung sucht möglichst kleine Vorhersagefehler

>* Verlust misst Modellfehler als eine Zahl
>* Parameterwerte erzeugen unterschiedliche Verluste

>* Ein Parameter macht Optimierung anschaulich.
>* Verlust zeigt den Weg zu besseren Werten.



In [ ]:
#@title Python-Code - Ein Parameter

# Wir untersuchen einen einzelnen Modellparameter.
# Der Verlust bewertet jede mögliche Parameterwahl.
# Die Grafik zeigt den besten Parameterwert.

import numpy as np
import matplotlib.pyplot as plt

# Kleine Beispieldaten zeigen Lernstunden und Prüfungspunkte.
study_hours = np.array([1, 2, 3, 4, 5], dtype=float)
exam_points = np.array([2, 4, 5, 8, 10], dtype=float)

# Diese Prüfung verhindert unpassende Datenlängen.
if study_hours.shape != exam_points.shape:
    raise ValueError("Die Daten müssen gleich viele Werte enthalten.")

# Wir testen viele mögliche Steigungen als einzigen Parameter.
parameter_values = np.linspace(0, 3, 121)
loss_values = []

# Für jede Steigung berechnen wir den mittleren quadratischen Fehler.
for parameter in parameter_values:
    predictions = parameter * study_hours
    errors = predictions - exam_points
    loss = np.mean(errors ** 2)
    loss_values.append(loss)

# Der kleinste Verlust markiert die beste getestete Steigung.
loss_values = np.array(loss_values)
best_index = np.argmin(loss_values)
best_parameter = parameter_values[best_index]
best_loss = loss_values[best_index]

print(f"Bester getesteter Parameter: {best_parameter:.2f} Punkte pro Stunde")
print(f"Kleinster Verlust: {best_loss:.2f}")
print("Ein niedriger Verlust bedeutet passendere Vorhersagen.")

# Die Kurve macht den Zusammenhang zwischen Parameter und Verlust sichtbar.
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(parameter_values, loss_values, label="Verlustfunktion")
ax.scatter(best_parameter, best_loss, color="red", label="kleinster Verlust")
ax.set_title("Verlustfunktion für einen Parameter")

ax.set_xlabel("Parameter: Punkte pro Lernstunde")
ax.set_ylabel("Mittlerer quadratischer Verlust")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()



### **1.2. Werteraster**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_07/Lecture_A/image_01_02.jpg?v=1784790283" width="250">



>* Parameterwerte systematisch ausprobieren
>* Verlust für jeden Wert vergleichen

>* Rasterpunkte zeigen konkrete Modellvarianten und Fehler
>* Optimierung vergleicht viele mögliche Parametereinstellungen

>* Feineres Raster bedeutet mehr Genauigkeit und Aufwand
>* Verlustform zeigt Richtung zum Minimum



In [ ]:
#@title Python-Code - Werteraster

# Dieses Beispiel zeigt ein Werteraster für einen Parameter.
# Jeder Rasterwert erhält einen eigenen mittleren quadratischen Verlust.
# Die Grafik macht das kleinste Verlustgebiet sichtbar.

import numpy as np
import matplotlib.pyplot as plt

# Kleine Beispieldaten beschreiben Wohnfläche und Monatsmiete.
area_sqm = np.array([30, 45, 60, 75, 90], dtype=float)
rent_eur = np.array([420, 560, 700, 860, 990], dtype=float)

# Wir halten den Achsenabschnitt fest und variieren nur die Steigung.
intercept_eur = 120.0
slope_grid = np.linspace(5.0, 12.0, 29)

# Für jede Steigung berechnen wir Vorhersagen und Verlust.
losses = []
for slope in slope_grid:
    predictions = intercept_eur + slope * area_sqm
    errors = predictions - rent_eur
    mean_squared_loss = np.mean(errors ** 2)
    losses.append(mean_squared_loss)

# Die Liste wird für Auswertung und Grafik umgewandelt.
losses = np.array(losses)
best_index = int(np.argmin(losses))
best_slope = slope_grid[best_index]

# Eine einfache Prüfung schützt vor unerwarteten Rasterproblemen.
if len(slope_grid) != len(losses):
    raise ValueError("Raster und Verluste müssen gleich lang sein.")

print(f"Geprüfte Steigungen: {len(slope_grid)}")
print(f"Beste Raster-Steigung: {best_slope:.2f} Euro pro Quadratmeter")
print(f"Kleinster Raster-Verlust: {losses[best_index]:.1f}")

# Die Kurve zeigt, wie der Verlust vom Parameter abhängt.
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(slope_grid, losses, marker="o", label="Rasterwerte")
ax.scatter(best_slope, losses[best_index], color="red", zorder=3, label="Minimum")

ax.set_title("Werteraster für eine Steigung")
ax.set_xlabel("Steigung in Euro pro Quadratmeter")
ax.set_ylabel("Mittlerer quadratischer Verlust")
ax.legend()

plt.show()



### **1.3. Lokale Steigung**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_07/Lecture_A/image_01_03.jpg?v=1784790288" width="250">



>* Steigung zeigt Verluständerung bei kleinen Parameterschritten
>* Sie weist die Richtung zur Verbesserung

>* Lokale Steigung zeigt bessere Parameter Richtung
>* Zu kleine oder große Werte korrigieren

>* Steigung gilt nur nahe am aktuellen Wert
>* Sie zeigt schrittweise sinnvolle Parameteränderungen



In [ ]:
#@title Python-Code - Lokale Steigung

# Wir untersuchen lokale Steigung einer Verlustfunktion.
# Ein Parameter bestimmt einfache lineare Vorhersagen.
# Die Grafik zeigt Richtung und Verluständerung.

import numpy as np
import matplotlib.pyplot as plt

# Kleine Beispieldaten beschreiben Wohnfläche und Heizkosten.
area = np.array([40, 55, 70, 85, 100], dtype=float)
cost = np.array([80, 105, 135, 165, 190], dtype=float)

# Skalierung hält die Zahlen für Anfänger übersichtlich.
area_scaled = area / 100.0
parameter_values = np.linspace(0, 260, 200)

# Diese Funktion berechnet den mittleren quadratischen Verlust.
def loss_for_parameter(parameter):
    predictions = parameter * area_scaled
    errors = predictions - cost
    return np.mean(errors ** 2)

# Wir prüfen eine kleine Änderung um den aktuellen Parameter.
current_parameter = 120.0
small_step = 1.0
current_loss = loss_for_parameter(current_parameter)

# Die lokale Steigung wird hier numerisch angenähert.
loss_after_step = loss_for_parameter(current_parameter + small_step)
local_slope = (loss_after_step - current_loss) / small_step
suggested_direction = "kleiner" if local_slope > 0 else "größer"

print(f"Aktueller Parameter: {current_parameter:.1f}")
print(f"Aktueller Verlust: {current_loss:.1f}")
print(f"Lokale Steigung: {local_slope:.1f}")
print(f"Hinweis: Parameter etwas {suggested_direction} wählen.")

# Für die Kurve berechnen wir den Verlust vieler Parameterwerte.
loss_values = np.array([loss_for_parameter(value) for value in parameter_values])
tangent_x = np.array([current_parameter - 25, current_parameter + 25])
tangent_y = current_loss + local_slope * (tangent_x - current_parameter)

# Die Grafik verbindet Verlustkurve und lokale Steigung.
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(parameter_values, loss_values, label="Verlustfunktion")
ax.plot(tangent_x, tangent_y, label="lokale Steigung")

ax.scatter([current_parameter], [current_loss], color="red", zorder=3)
ax.set_title("Lokale Steigung bei einem Modellparameter")
ax.set_xlabel("Parameterwert: Kosten pro skalierter Wohnfläche")
ax.set_ylabel("Mittlerer quadratischer Verlust")

ax.legend()
ax.grid(True, alpha=0.3)
plt.show()



## **2. Gradientenabstieg verstehen**

### **2.1. Eindimensionale Funktion**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_07/Lecture_A/image_02_01.jpg?v=1784790273" width="250">



>* Ein Parameter bestimmt den Verlustwert.
>* Lokale Steigung zeigt die bessere Richtung.

>* Ableitung zeigt Richtung und Stärke
>* Lokale Schritte führen zum Minimum

>* Schrittweise Richtung geringeren Verlusts gehen
>* Iterativ aus Fehlern bessere Lösungen finden



In [ ]:
#@title Python-Code - Eindimensionale Funktion

# Dieses Beispiel zeigt Gradientenabstieg in einer Dimension.
# Die Ableitung bestimmt Richtung und Schrittgröße.
# Die Grafik zeigt Schritte zum Minimum.

import numpy as np
import matplotlib.pyplot as plt

# Diese Verlustfunktion hat ihr Minimum bei x gleich zwei.
def loss(x):
    return (x - 2) ** 2 + 1

# Die Ableitung beschreibt die lokale Steigung der Funktion.
def gradient(x):
    return 2 * (x - 2)

# Wir prüfen eine einfache Annahme vor der Optimierung.
learning_rate = 0.25
if learning_rate <= 0:
    raise ValueError("Die Lernrate muss positiv sein.")

# Der Startpunkt liegt absichtlich weit vom Minimum entfernt.
current_x = -4.0
steps = [current_x]
losses = [loss(current_x)]

# Jeder Schritt geht entgegen der aktuellen Steigung.
for step in range(12):
    current_gradient = gradient(current_x)
    current_x = current_x - learning_rate * current_gradient
    steps.append(current_x)
    losses.append(loss(current_x))

# Kurze Zahlen zeigen die Annäherung ohne lange Tabellen.
print(f"Startwert: x = {steps[0]:.2f}, Verlust = {losses[0]:.2f}")
print(f"Endwert: x = {steps[-1]:.2f}, Verlust = {losses[-1]:.4f}")
print(f"Bekanntes Minimum: x = 2.00, Verlust = 1.00")

# Die Kurve zeigt Verlustlandschaft und besuchte Punkte.
x_values = np.linspace(-5, 5, 300)
y_values = loss(x_values)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(x_values, y_values, label="Verlustfunktion")
ax.scatter(steps, losses, color="red", label="Gradientenabstieg")

# Pfeile machen die Bewegungsrichtung sichtbar.
for index in range(len(steps) - 1):
    ax.annotate(
        "",
        xy=(steps[index + 1], losses[index + 1]),
        xytext=(steps[index], losses[index]),
        arrowprops={"arrowstyle": "->", "color": "red"},
    )

ax.set_title("Gradientenabstieg für eine eindimensionale Funktion")
ax.set_xlabel("Parameterwert x")
ax.set_ylabel("Verlust")
ax.legend()
plt.show()



### **2.2. Lernrate wählen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_07/Lecture_A/image_02_02.jpg?v=1784790271" width="250">



>* Lernrate steuert die Schrittgröße zum Minimum
>* Zu groß oder klein macht Optimierung problematisch

>* Zu kleine Lernrate verlangsamt das Lernen.
>* Zu große Lernrate macht Schritte instabil.

>* Lernrate durch Verlustkurven gezielt prüfen
>* Schrittweite an die Verlustlandschaft anpassen



In [ ]:
#@title Python-Code - Lernrate wählen

# Dieses Beispiel vergleicht mehrere Lernraten.
# Gradientenabstieg sucht das Minimum einer Parabel.
# Die Kurven zeigen stabile und instabile Schritte.

import numpy as np
import matplotlib.pyplot as plt

# Diese einfache Verlustfunktion hat ihr Minimum bei drei.
def loss(parameter):
    return (parameter - 3) ** 2

# Der Gradient zeigt die lokale Steigung der Parabel.
def gradient(parameter):
    return 2 * (parameter - 3)

# Jede Lernrate startet beim gleichen Parameterwert.
learning_rates = [0.05, 0.3, 1.05]
start_parameter = -4.0
steps = 18

# Wir speichern die Verlustwerte für jede Lernrate.
histories = {}
for learning_rate in learning_rates:
    parameter = start_parameter
    losses = []

    for step in range(steps + 1):
        losses.append(loss(parameter))
        parameter = parameter - learning_rate * gradient(parameter)

    histories[learning_rate] = losses

# Kurze Zahlen helfen beim Lesen der Kurven.
for learning_rate in learning_rates:
    first_loss = histories[learning_rate][0]
    final_loss = histories[learning_rate][-1]
    print(f"Lernrate {learning_rate}: Start {first_loss:.2f}, Ende {final_loss:.2f}")

# Eine einzelne Grafik vergleicht die Verlustkurven direkt.
fig, ax = plt.subplots(figsize=(8, 5))
for learning_rate, losses in histories.items():
    ax.plot(range(steps + 1), losses, marker="o", label=f"Lernrate {learning_rate}")

ax.set_title("Einfluss der Lernrate auf den Gradientenabstieg")
ax.set_xlabel("Schritt")
ax.set_ylabel("Verlust")
ax.legend()
ax.grid(True, alpha=0.3)

plt.show()



### **2.3. Sinnvoll stoppen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_07/Lecture_A/image_02_03.jpg?v=1784790275" width="250">



>* Stoppen, wenn kaum Fortschritt entsteht
>* Verlust, Parameteränderung und Steigung prüfen

>* Maximale Iterationen verhindern endloses Weiterlaufen
>* Stopp nach Nutzen und Rechenaufwand wählen

>* Stoppkriterien mit Lernrate und Skalierung prüfen
>* Verlustkurve überwachen, statt blind weiterzurechnen



In [ ]:
#@title Python-Code - Sinnvoll stoppen

# Wir stoppen Gradientenabstieg mit klaren Kriterien.
# Kleine Verbesserungen zeigen Nähe zum Minimum.
# Die Grafik vergleicht sinnvolle Abbruchpunkte.

import numpy as np
import matplotlib.pyplot as plt

# Diese einfache Verlustfunktion hat ihr Minimum bei drei.
def loss(parameter):
    return (parameter - 3.0) ** 2 + 1.0

# Die Ableitung zeigt die Richtung des steilsten Anstiegs.
def gradient(parameter):
    return 2.0 * (parameter - 3.0)

learning_rate = 0.2
max_iterations = 40
loss_tolerance = 0.0001
parameter_tolerance = 0.0001

parameter = -4.0
parameters = [parameter]
losses = [loss(parameter)]
stop_reason = "maximale Iterationen erreicht"

# Jeder Schritt prüft Fortschritt, Bewegung und Steigung.
for iteration in range(1, max_iterations + 1):
    old_parameter = parameter
    old_loss = loss(old_parameter)
    parameter = old_parameter - learning_rate * gradient(old_parameter)

    new_loss = loss(parameter)
    parameters.append(parameter)
    losses.append(new_loss)

    loss_change = abs(old_loss - new_loss)
    parameter_change = abs(old_parameter - parameter)
    slope_size = abs(gradient(parameter))

    if loss_change < loss_tolerance:
        stop_reason = "Verlust verbessert sich kaum noch"
        break

    if parameter_change < parameter_tolerance:
        stop_reason = "Parameter bewegt sich kaum noch"
        break

    if slope_size < parameter_tolerance:
        stop_reason = "Steigung ist nahe null"
        break

if len(parameters) != len(losses):
    raise ValueError("Parameter und Verlustwerte passen nicht zusammen.")

print(f"Stopp nach {len(losses) - 1} Iterationen: {stop_reason}.")
print(f"Gefundener Parameter: {parameter:.4f}.")
print(f"Letzter Verlust: {losses[-1]:.6f}.")

# Die Kurve zeigt, wann weitere Schritte wenig bringen.
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(len(losses)), losses, marker="o", label="Verlust")
ax.axhline(1.0, color="gray", linestyle="--", label="Minimum")

ax.set_title("Sinnvoll stoppen beim Gradientenabstieg")
ax.set_xlabel("Iteration")
ax.set_ylabel("Verlust")
ax.legend()
ax.grid(True, alpha=0.3)

plt.show()



## **3. Konvergenz prüfen**

### **3.1. Epochen und Verlauf**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_07/Lecture_A/image_03_01.jpg?v=1784790277" width="250">



>* Epochen zeigen wiederholte Modellverbesserungen
>* Verlustverlauf zeigt Konvergenzprobleme

>* Verlustkurven zeigen Lernmuster über Epochen.
>* Verläufe immer im Kontext der Einstellungen deuten.

>* Konvergenz nach Tempo und Stabilität bewerten
>* Training rechtzeitig, aber nicht zu früh beenden



In [ ]:
#@title Python-Code - Epochen und Verlauf

# Wir beobachten Verlustwerte über mehrere Epochen.
# Die Lernrate beeinflusst Stabilität und Geschwindigkeit.
# Die Kurve zeigt Konvergenz oder Probleme.

import numpy as np
import matplotlib.pyplot as plt

# Kleine Daten simulieren eine einfache lineare Regression.
x_values = np.array([1, 2, 3, 4, 5], dtype=float)
y_values = np.array([2, 4, 6, 8, 10], dtype=float)

# Diese Prüfung verhindert unpassende Datenformen.
if x_values.shape != y_values.shape:
    raise ValueError("x_values und y_values müssen gleich lang sein.")

# Wir optimieren nur die Steigung bei festem Achsenabschnitt.
learning_rate = 0.08
slope = 0.0
loss_history = []

# Jede Epoche berechnet Verlust und verbessert die Steigung.
for epoch in range(40):
    predictions = slope * x_values
    errors = predictions - y_values
    loss = np.mean(errors ** 2)
    loss_history.append(loss)

    gradient = 2 * np.mean(errors * x_values)
    slope = slope - learning_rate * gradient

# Kurze Kennzahlen helfen beim Lesen der Verlustkurve.
print(f"Startverlust: {loss_history[0]:.2f}")
print(f"Endverlust: {loss_history[-1]:.6f}")
print(f"Gelernte Steigung: {slope:.4f}")

# Die Verlustkurve macht den Verlauf über Epochen sichtbar.
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(1, len(loss_history) + 1), loss_history, marker="o")
ax.set_title("Verlustkurve beim Gradientenabstieg")
ax.set_xlabel("Epoche")
ax.set_ylabel("Mittlerer quadratischer Fehler")
ax.grid(True, alpha=0.3)
plt.show()



### **3.2. Skalierung und Konvergenz**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_07/Lecture_A/image_03_02.jpg?v=1784790279" width="250">



>* Skalierung macht Optimierung gleichmäßiger.
>* Große Werte können Gradientenabstieg dominieren.

>* Skalierung macht Gradientenabstieg stabiler und schneller
>* Ähnliche Skalen ermöglichen passendere Lernraten

>* Konvergenz immer im Zusammenspiel bewerten
>* Verlustkurven zeigen Lernen, Stagnation oder Instabilität



In [ ]:
#@title Python-Code - Skalierung und Konvergenz

# Dieses Beispiel zeigt Skalierung beim Gradientenabstieg.
# Unterschiedliche Skalen erschweren stabile Konvergenz.
# Die Verlustkurven machen den Effekt sichtbar.

import numpy as np
import matplotlib.pyplot as plt

# Wir erzeugen zwei Merkmale mit sehr verschiedenen Größenordnungen.
rng = np.random.default_rng(42)
sample_count = 80
small_feature = rng.uniform(0.0, 1.0, sample_count)
large_feature = rng.uniform(0.0, 1000.0, sample_count)

# Das Ziel folgt einer einfachen linearen Beziehung.
noise = rng.normal(0.0, 0.5, sample_count)
y = 3.0 * small_feature + 0.01 * large_feature + noise
X_unscaled = np.column_stack((small_feature, large_feature))

# Diese Prüfung verhindert unpassende Formen im Beispiel.
if X_unscaled.shape != (sample_count, 2):
    raise ValueError("Die Merkmalsmatrix hat nicht die erwartete Form.")

# Standardisierung bringt beide Merkmale auf vergleichbare Skalen.
feature_means = X_unscaled.mean(axis=0)
feature_stds = X_unscaled.std(axis=0)
X_scaled = (X_unscaled - feature_means) / feature_stds

# Diese Funktion führt einfachen Gradientenabstieg aus.
def gradient_descent_losses(X, y, learning_rate, steps):
    weights = np.zeros(X.shape[1])
    losses = []
    for step in range(steps):
        predictions = X @ weights
        errors = predictions - y
        loss = np.mean(errors ** 2)
        losses.append(loss)
        gradient = (2.0 / len(y)) * (X.T @ errors)
        weights = weights - learning_rate * gradient
    return np.array(losses)

# Dieselbe Lernrate wirkt ohne Skalierung instabiler.
steps = 60
learning_rate = 0.1
unscaled_losses = gradient_descent_losses(X_unscaled, y, learning_rate, steps)
scaled_losses = gradient_descent_losses(X_scaled, y, learning_rate, steps)

# Wir begrenzen extreme Werte nur für eine lesbare Darstellung.
plot_limit = 200.0
unscaled_plot = np.minimum(unscaled_losses, plot_limit)
scaled_plot = np.minimum(scaled_losses, plot_limit)

print("Lernrate:", learning_rate)
print("Startverlust unskaliert:", round(float(unscaled_losses[0]), 2))
print("Endverlust unskaliert:", round(float(unscaled_losses[-1]), 2))
print("Startverlust skaliert:", round(float(scaled_losses[0]), 2))
print("Endverlust skaliert:", round(float(scaled_losses[-1]), 2))

# Die Kurven zeigen, wie Skalierung die Konvergenz beruhigt.
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(unscaled_plot, label="ohne Skalierung")
ax.plot(scaled_plot, label="mit Standardisierung")
ax.set_title("Skalierung und Konvergenz beim Gradientenabstieg")

ax.set_xlabel("Iterationsschritt")
ax.set_ylabel("Mittlerer quadratischer Verlust")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()



### **3.3. Optimierung praktisch üben**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_07/Lecture_A/image_03_03.jpg?v=1784790281" width="250">



>* Optimierung als Ausprobieren und Anpassen verstehen
>* Verlustkurven zeigen passende Lernraten

>* Abbruchzeitpunkt fachlich sinnvoll wählen
>* Verlauf kritisch auf Konvergenz prüfen

>* Skalierung macht Gradientenabstieg stabiler
>* Verlustkurven zeigen passende Anpassungen



In [ ]:
#@title Python-Code - Optimierung praktisch üben

# Wir prüfen Konvergenz beim Gradientenabstieg.
# Lernrate und Skalierung verändern Verlustkurven.
# Die Grafik zeigt stabile und instabile Verläufe.

import numpy as np
import matplotlib.pyplot as plt

# Diese kleinen Daten beschreiben Wohnfläche und Preis.
area_m2 = np.array([45, 55, 65, 75, 85, 95], dtype=float)
price_k = np.array([180, 210, 250, 285, 320, 360], dtype=float)

# Eine einfache Prüfung verhindert unpassende Datenformen.
if area_m2.shape != price_k.shape:
    raise ValueError("Wohnfläche und Preis brauchen gleich viele Werte.")

# Skalierung macht die Eingabe für Optimierung ausgeglichener.
area_scaled = (area_m2 - area_m2.mean()) / area_m2.std()
price_centered = price_k - price_k.mean()

# Diese Funktion optimiert nur eine Steigung.
def run_gradient_descent(feature, target, learning_rate, steps):
    slope = 0.0
    losses = []
    for step in range(steps):
        prediction = slope * feature
        error = prediction - target
        loss = np.mean(error ** 2)
        gradient = 2 * np.mean(error * feature)
        losses.append(loss)
        slope = slope - learning_rate * gradient
    return np.array(losses), slope

# Drei Einstellungen zeigen typische Konvergenzprobleme.
loss_scaled_good, slope_good = run_gradient_descent(
    area_scaled, price_centered, 0.25, 40
)

loss_scaled_fast, slope_fast = run_gradient_descent(
    area_scaled, price_centered, 1.05, 40
)

loss_unscaled, slope_unscaled = run_gradient_descent(
    area_m2, price_centered, 0.00001, 40
)

# Kurze Kennzahlen helfen beim Lesen der Kurven.
print("Gute Lernrate, skalierte Daten: Verlust", round(loss_scaled_good[-1], 2))
print("Zu große Lernrate, skalierte Daten: Verlust", round(loss_scaled_fast[-1], 2))
print("Kleine Lernrate, unskalierte Daten: Verlust", round(loss_unscaled[-1], 2))
print("Abbruch nach 40 Schritten: weitere Verbesserung prüfen.")

# Eine einzige Grafik vergleicht die Verlustkurven direkt.
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(loss_scaled_good, label="skaliert, Lernrate 0.25")
ax.plot(loss_scaled_fast, label="skaliert, Lernrate 1.05")
ax.plot(loss_unscaled, label="unskaliert, Lernrate 0.00001")

ax.set_title("Konvergenz anhand von Verlustkurven prüfen")
ax.set_xlabel("Optimierungsschritt")
ax.set_ylabel("Mittlerer quadratischer Verlust")
ax.legend()

plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Optimierung verstehen**</font>


In this lecture, you learned to:
- Beschreiben eine Verlustfunktion für einen einzelnen Modellparameter. 
- Implementieren Raster- und Gradientenabstiegssuche für einfache Funktionen. 
- Analysieren Lernrate, Abbruchbedingungen, Skalierung und Verlustkurven. 

In the next Lecture (Lecture B), we will go over 'Regression mit NumPy'